# Knowledge Distillation: see the dark knowledge, then prove KD beats hard-only

A teacher's *softened* output distribution carries **dark knowledge** — the relative probabilities it assigns to the *wrong* classes, which encode how categories relate. A one-hot hard label throws all of that away. This notebook makes the idea concrete, end to end, on a tiny synthetic 5-class task:

1. **Soft targets** — print one teacher output at `T=1` vs `T=4` so the dark knowledge becomes *visible*.
2. **The KD loss** — implement `L = alpha * T^2 * KL(teacher_T || student_T) + (1-alpha) * CE`, with the temperature division, the KL term, and the `T^2` factor each explained inline.
3. **The payoff** — train a narrow student two ways (hard-labels-only vs KD) from an *identical* init and show the KD student agrees with the teacher more.

It runs on CPU in a few seconds. The code matches `knowledge_distillation.py` next to this notebook. The companion page is [11-Knowledge-Distillation.md](../11-Knowledge-Distillation.md).

> **Step 1 — imports, hyperparameters, and honest device detection.** The two hyperparameters that *define* a KD run are the temperature `T` and the mix weight `alpha`; we hoist them as named constants. We **detect** the best accelerator but **pin** the reproducible run to CPU and print so honestly — the small nets + fixed seed make CPU both fast and deterministic.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# The two hyperparameters that define a KD run.
TEMPERATURE = 4.0   # T: softmax temperature; T>1 flattens the distribution to expose dark knowledge
ALPHA = 0.9         # weight on the soft (distillation) term; (1-alpha) on the hard labels

# Problem + model sizes.
N_CLASSES, N_FEATURES = 5, 20
NOISE_STD = 0.55          # small blob spread -> a strong teacher is achievable -> real dark knowledge
N_TEACHER_TRAIN = 4_000   # ABUNDANT data for the teacher
N_STUDENT_TRAIN = 150     # a SMALL labelled subset for the students (the data-starved regime KD shines in)
N_TEST = 2_000
TEACHER_HIDDEN, STUDENT_HIDDEN = 256, 8   # 32x narrower student
TEACHER_EPOCHS, STUDENT_EPOCHS = 200, 300
LEARNING_RATE = 0.01
SEED = 0

# Detect, but pin the reproducible trace to CPU and say so.
DETECTED = ('cuda' if torch.cuda.is_available()
            else 'mps' if torch.backends.mps.is_available() else 'cpu')
DEVICE = torch.device('cpu')
print(f'device: {DEVICE} (detected {DETECTED}; pinned to CPU for reproducibility)')
print('torch:', torch.__version__)

device: cpu (detected mps; pinned to CPU for reproducibility)
torch: 2.12.0


> **Step 2 — a synthetic task with built-in class similarity.** Each class is a Gaussian blob around a mean vector. We deliberately place class 0 (**cat**) and class 1 (**dog**) *close together*, so a good teacher learns they are confusable — that confusability is the dark knowledge we want to transfer. The other classes sit far apart (distinct).

In [2]:
def make_data(n, gen):
    means = torch.zeros(N_CLASSES, N_FEATURES, device=DEVICE)
    means[0, :3] = torch.tensor([2.0, 2.0, 0.0], device=DEVICE)   # cat
    means[1, :3] = torch.tensor([2.0, 1.4, 0.5], device=DEVICE)   # dog: close to cat on purpose
    means[2, :3] = torch.tensor([-2.0, 2.0, 0.0], device=DEVICE)  # distinct
    means[3, :3] = torch.tensor([0.0, -2.5, 1.0], device=DEVICE)  # distinct
    means[4, :3] = torch.tensor([-2.0, -2.0, -1.0], device=DEVICE)  # distinct
    labels = torch.randint(0, N_CLASSES, (n,), generator=gen, device=DEVICE)
    feats = means[labels] + torch.randn(n, N_FEATURES, generator=gen, device=DEVICE) * NOISE_STD
    return feats, labels

gen = torch.Generator(device=DEVICE).manual_seed(SEED)
teacher_x, teacher_y = make_data(N_TEACHER_TRAIN, gen)   # abundant
student_x, student_y = make_data(N_STUDENT_TRAIN, gen)   # small subset
test_x, test_y = make_data(N_TEST, gen)
print('teacher train:', tuple(teacher_x.shape), ' student train:', tuple(student_x.shape),
      ' test:', tuple(test_x.shape))

teacher train: (4000, 20)  student train: (150, 20)  test: (2000, 20)


> **Step 3 — one model class; width is the only difference.** Teacher and student are the same one-hidden-layer MLP — the teacher just much **wider** (256 vs 8 hidden units). The final layer returns **raw logits** (no softmax): distillation operates on logits, because that is what we soften with temperature.

In [3]:
class MLP(nn.Module):
    def __init__(self, hidden):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(N_FEATURES, hidden), nn.ReLU(),
            nn.Linear(hidden, N_CLASSES),   # returns LOGITS (no softmax) -- KD needs logits
        )
    def forward(self, x):
        return self.net(x)

def accuracy(model, x, y):
    with torch.no_grad():
        return (model(x).argmax(-1) == y).float().mean().item()

> **Step 4 — train the wide teacher on abundant data.** Plain cross-entropy on the hard labels until it is accurate. Re-seed first so the teacher's init is fixed across runs. Its test accuracy is the **ceiling** the small student will chase.

In [4]:
torch.manual_seed(SEED)
teacher = MLP(TEACHER_HIDDEN).to(DEVICE)
opt = torch.optim.Adam(teacher.parameters(), lr=LEARNING_RATE)
for _ in range(TEACHER_EPOCHS):
    opt.zero_grad()
    F.cross_entropy(teacher(teacher_x), teacher_y).backward()
    opt.step()
teacher.eval()
print(f'teacher test accuracy: {accuracy(teacher, test_x, test_y):.3f}  '
      f'(wide net: {TEACHER_HIDDEN} hidden units, {N_TEACHER_TRAIN} train pts)')

teacher test accuracy: 0.872  (wide net: 256 hidden units, 4000 train pts)


## See the dark knowledge

This is the single most important thing to *see*. We take one **cat** example the teacher gets right, and print its output two ways: the ordinary `T=1` softmax, and the softened `T=4` softmax. At `T=1` the teacher is ~95% sure and the off-target mass is invisible; at `T=4` the *same logits* reveal a clear "cat, but quite dog-like, a bit lynx-like, nothing like car/plane." That emerging structure is the dark knowledge.

> **Step 5 — softening one teacher output.** We pick a correctly-classified cat that the teacher is *not maximally saturated* on (a mid-confidence one), so the softened dog/lynx mass is visible rather than rounded to zero — representative, not cherry-picked.

In [5]:
with torch.no_grad():
    logits_all = teacher(teacher_x)
correct_cat = (teacher_y == 0) & (logits_all.argmax(-1) == 0)   # cats the teacher gets right
cat_logits = logits_all[correct_cat]
conf = F.softmax(cat_logits, dim=-1)[:, 0]                       # P(cat) at T=1
idx = conf.argsort()[len(conf) // 4]                            # 25th-percentile confidence
logits = cat_logits[idx]

names = ['cat', 'dog', 'lynx', 'car', 'plane']
p1 = F.softmax(logits, dim=-1)                  # T=1: the sharp distribution
p4 = F.softmax(logits / TEMPERATURE, dim=-1)    # T=4: the softened distribution KD uses
print(f"{'class':>8} | {'T=1 prob':>9} | {'T=4 prob':>9}")
print('-' * 32)
for nm, a, b in zip(names, p1.tolist(), p4.tolist()):
    print(f'{nm:>8} | {a:>9.3f} | {b:>9.3f}')
print(f"-> at T=1 'dog' carries {p1[1]:.3f}; at T=4 it lifts to {p4[1]:.3f} "
      f'(the dark knowledge a one-hot label discards)')

   class |  T=1 prob |  T=4 prob
--------------------------------
     cat |     0.953 |     0.630
     dog |     0.047 |     0.298
    lynx |     0.000 |     0.066
     car |     0.000 |     0.005
   plane |     0.000 |     0.000
-> at T=1 'dog' carries 0.047; at T=4 it lifts to 0.298 (the dark knowledge a one-hot label discards)


## The KD loss

Now the heart of it: `L = alpha * T^2 * KL(teacher_T || student_T) + (1-alpha) * CE(student, hard)`.
Three things to watch in the code: (1) we divide logits by `T` **before** softmax (the softening); (2) the **KL term** pulls the student's whole softened distribution onto the teacher's; (3) the **`T^2` factor** cancels the `1/T^2` shrink of the soft gradient, so raising `T` doesn't silently turn the soft term off.

> **Step 6 — implement the distillation loss.** `kl_div` expects the *input* as log-probs and the *target* as probs, so we use `log_softmax` for the student and `softmax` for the teacher. `reduction='batchmean'` is the mathematically correct KL average over the batch.

In [6]:
def distillation_loss(student_logits, teacher_logits, hard_labels,
                      temperature=TEMPERATURE, alpha=ALPHA):
    # soft term: soften both distributions by T, then match them with KL
    student_log_soft = F.log_softmax(student_logits / temperature, dim=-1)  # student_T (log-probs)
    teacher_soft     = F.softmax(teacher_logits / temperature, dim=-1)      # teacher_T (the target)
    soft_kl = F.kl_div(student_log_soft, teacher_soft, reduction='batchmean')
    soft_term = (temperature ** 2) * soft_kl   # T^2: cancels the 1/T^2 gradient shrink
    # hard term: ordinary cross-entropy on the true labels (at T=1)
    hard_ce = F.cross_entropy(student_logits, hard_labels)
    return alpha * soft_term + (1.0 - alpha) * hard_ce

# sanity check on the current (untrained-ish) shapes
demo = distillation_loss(torch.randn(4, N_CLASSES), torch.randn(4, N_CLASSES),
                         torch.randint(0, N_CLASSES, (4,)))
print('distillation_loss on a 4-example batch ->', float(demo))

distillation_loss on a 4-example batch -> 1.0526624917984009


## Train two students from an identical start

The fair experiment: build **one** student initialization, clone it, and train two copies that differ **only** in the loss — one on hard labels, one with KD. Same architecture, same starting weights, same data (the small 150-point subset). Any difference in the result is due to the loss alone.

> **Step 7 — a shared student init.** Seed (distinctly from the teacher), build the student, and snapshot its weights so both training runs start identically.

In [7]:
torch.manual_seed(SEED + 1)
init_student = MLP(STUDENT_HIDDEN).to(DEVICE)
init_state = {k: v.clone() for k, v in init_student.state_dict().items()}
print('shared student init captured:', list(init_state.keys()))

shared student init captured: ['net.0.weight', 'net.0.bias', 'net.2.weight', 'net.2.bias']


> **Step 8a — baseline: hard labels only.** Train the student on plain cross-entropy — no teacher, no dark knowledge. This is what you get *without* distillation.

In [8]:
student_hard = MLP(STUDENT_HIDDEN).to(DEVICE)
student_hard.load_state_dict(init_state)         # identical starting weights
opt = torch.optim.Adam(student_hard.parameters(), lr=LEARNING_RATE)
for _ in range(STUDENT_EPOCHS):
    opt.zero_grad()
    F.cross_entropy(student_hard(student_x), student_y).backward()  # hard labels only
    opt.step()
student_hard.eval()
print('hard-only student trained.')

hard-only student trained.


> **Step 8b — the KD student.** Same init, same data — but the loss is the distillation loss. The teacher's logits on the student's data are fixed targets, so we compute them once up front, then train the student to match the softened teacher (plus the hard labels).

In [9]:
student_kd = MLP(STUDENT_HIDDEN).to(DEVICE)
student_kd.load_state_dict(init_state)           # SAME starting weights as the hard-only student
opt = torch.optim.Adam(student_kd.parameters(), lr=LEARNING_RATE)
with torch.no_grad():
    teacher_logits = teacher(student_x)          # fixed targets -> compute once
for _ in range(STUDENT_EPOCHS):
    opt.zero_grad()
    distillation_loss(student_kd(student_x), teacher_logits, student_y).backward()  # the KD objective
    opt.step()
student_kd.eval()
print('KD student trained.')

KD student trained.


## The payoff: does the student behave like the teacher?

The KD-specific metric is **agreement** — not "is the student right" but "does the student's prediction match the *teacher's* prediction." That is precisely what distillation tries to transfer. We assert KD agreement is at least as high as hard-only — the claim of the whole topic, checked, not assumed.

> **Step 9 — measure accuracy and teacher-agreement, then assert.** Both students saw the same 150 points from the same init; only the loss differed. We expect the KD student to agree with the teacher more — and, in this data-starved regime, to be more accurate too.

In [10]:
def agreement(student, teacher, x):
    with torch.no_grad():
        return (student(x).argmax(-1) == teacher(x).argmax(-1)).float().mean().item()

hard_acc, kd_acc = accuracy(student_hard, test_x, test_y), accuracy(student_kd, test_x, test_y)
hard_agree, kd_agree = agreement(student_hard, teacher, test_x), agreement(student_kd, teacher, test_x)
teacher_acc = accuracy(teacher, test_x, test_y)

print(f"{'student':>16} | {'test acc':>9} | {'agreement w/ teacher':>21}")
print('-' * 54)
print(f"{'hard-labels-only':>16} | {hard_acc:>9.3f} | {hard_agree:>21.3f}")
print(f"{'KD (soft+hard)':>16} | {kd_acc:>9.3f} | {kd_agree:>21.3f}")
print(f"{'teacher (ref)':>16} | {teacher_acc:>9.3f} | {'1.000':>21}")

assert kd_agree >= hard_agree, f'KD agreement {kd_agree:.3f} should be >= hard-only {hard_agree:.3f}'
print(f'\nPASS: KD student agreement {kd_agree:.3f} >= hard-only {hard_agree:.3f} '
      f'(+{(kd_agree - hard_agree) * 100:.1f} points) -- distillation transferred the '
      f"teacher's behaviour, not just the labels.")

         student |  test acc |  agreement w/ teacher
------------------------------------------------------
hard-labels-only |     0.809 |                 0.812
  KD (soft+hard) |     0.882 |                 0.887
   teacher (ref) |     0.872 |                 1.000

PASS: KD student agreement 0.887 >= hard-only 0.812 (+7.6 points) -- distillation transferred the teacher's behaviour, not just the labels.


## Try it yourself

Before you change anything, **predict** each outcome, then run:

1. **Raise `N_STUDENT_TRAIN` to 4000** (give the student abundant data). Does the KD-vs-hard gap *grow* or *shrink*? *(Hint: distillation's benefit is largest when the student is data- or capacity-limited; with plenty of data the hard-only baseline catches up.)*
2. **Set `ALPHA = 1.0`** (pure soft targets, no hard anchor). Does test accuracy go up or down? *(The hard term anchors the student to ground truth where the teacher is wrong.)*
3. **Delete the `T^2` factor** (replace `(temperature ** 2) * soft_kl` with `soft_kl`). Watch accuracy *drop* — the soft term is now `T^2 = 16x` too weak. This is the single most common KD bug.
4. **Set `TEMPERATURE = 1.0`** (no softening). The dark knowledge collapses to near-one-hot and KD's edge over hard-only mostly vanishes — because there is no extra structure to transfer.

## What we saw

- **Dark knowledge is real and temperature-gated.** At `T=1` the teacher's "cat" output hid the dog/lynx structure; at `T=4` the same logits revealed it. That structure is what a one-hot label can never provide.
- **The KD loss is soft-KL + hard-CE, with a `T^2` factor.** The `T^2` keeps the soft term's gradient comparable to the hard term's as you change `T` — forget it and distillation silently weakens.
- **KD transferred the teacher's behaviour.** From an identical init and the same tiny labelled set, the KD student agreed with the teacher markedly more than the hard-only student (+7.6 points) — and, in this data-starved regime, was more accurate too.

That is the whole mechanism behind DistilBERT (40% smaller, ~97% of the quality) and the modern small-LLM wave — applied over a vocabulary instead of five classes. See the [concept page](../11-Knowledge-Distillation.md) for the flavors, the LLM-specific forms, and the limits.